In [1]:
#Project Checkpoint 4
#Group 1

In [2]:
#import libraries
import re
import pandas as pd
from pathlib import Path

In [ ]:
#define inut file and output folder
INPUT_FILE = Path("Project Checkpoint 4 Sample Data.xlsx")
OUTPUT_DIR = Path("project_checkpoint4_cleaned_files")

In [4]:
#columns contain dates or datetime values
DATE_COLUMNS = {
    "expense_date",
    "sale_datetime",
    "delivery_date",
    "order_date",
    "expected_delivery_date",
    "adjustment_date",
    "last_updated",
    "shift_date",
    "hire_date",
    "opening_date",
    "date_assigned"
}

#columns contain time values
TIME_COLUMNS = {
    "start_time",
    "end_time"
}

#email addresses should be stored in lowercase.
LOWERCASE_COLUMNS = {
    "email"
}

#state abbreviations and laptop serial numbers should be stored in uppercase.
UPPERCASE_COLUMNS = {
    "state",
    "serial_number"
}

#text columns should use title capitalization
TITLE_CASE_COLUMNS = {
    "first_name",
    "last_name",
    "contact_name",
    "department_name",
    "category_name",
    "expense_type",
    "payment_method",
    "adjustment_type",
    "job_title",
    "manufacturer",
    "brand",
    "city",
    "vendor_name"
}

#columns contain status descriptions
STATUS_COLUMNS = {
    "employment_status",
    "operating_status",
    "schedule_status",
    "delivery_status",
    "order_status",
    "vendor_status"
}

#create consistent  values
STATUS_MAP = {
    "full time": "Full-Time",
    "full-time": "Full-Time",
    "part time": "Part-Time",
    "part-time": "Part-Time",
    "in progress": "In Progress",
    "in-progress": "In Progress"
}

In [5]:
#convert names to snake_case
def snake_case(value):
    #convert the value to text and remove leading or trailing spaces
    text = str(value).strip()
    #replace spaces and special characters with underscores
    text = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        text
    )

    #remove underscores from the beginning or end and convert the value to lowercase
    return text.strip("_").lower()

#clean a pandas text column
def normalize_text(series):
    #convert all values to pandas string format and trim spaces from the beginning and end
    cleaned_series = (
        series
        .astype("string")
        .str.strip()
    )

    #replace two or more spaces with one space
    cleaned_series = cleaned_series.str.replace(
        r"\s+",
        " ",
        regex=True
    )

    #convert empty or text-based missing values into pd.NA.
    cleaned_series = cleaned_series.replace(
        {
            "": pd.NA,
            "nan": pd.NA,
            "None": pd.NA
        }
    )

    return cleaned_series

#read one Excel worksheet and keep only the raw data table
def extract_data_table(file_path, sheet_name):
    raw_sheet = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        header=None
    )

    header_row = raw_sheet.iloc[1].tolist()
    ending_column = len(header_row)

    for column_index, header_value in enumerate(header_row):

        if column_index > 0:
            #check whether the header is missing or empty
            if (
                pd.isna(header_value)
                or str(header_value).strip() == ""
            ):
                ending_column = column_index
                break

    #convert all retained column headers to snake_case
    cleaned_headers = [
        snake_case(header_value)
        for header_value in header_row[:ending_column]
    ]

    #select only the rows below the header and only the columns that belong to the raw data table
    data = raw_sheet.iloc[
        2:,
        :ending_column
    ].copy()

    #assign the cleaned column names
    data.columns = cleaned_headers

    #remove rows where every value is missing
    data = data.dropna(
        how="all"
    )

    if cleaned_headers:

        first_column = cleaned_headers[0]

        data = data[
            data[first_column]
            .astype("string")
            .str.lower()
            != first_column.lower()
        ]

    #reset the row index so it begins at zero
    return data.reset_index(
        drop=True
    )

#apply reusable cleaning rules to a table
def clean_table(dataframe):
    #create a copy 
    cleaned = dataframe.copy()

    #remove columns where every value is missing
    cleaned = cleaned.dropna(
        axis=1,
        how="all"
    )

    #standardize all column names using snake_case
    cleaned.columns = [
        snake_case(column)
        for column in cleaned.columns
    ]

    for column in cleaned.columns:
        column_name = snake_case(column)

        #clean date and datetime columns
        if (
            column_name in DATE_COLUMNS
            or column_name.endswith("_date")
            or column_name.endswith("_datetime")
        ):
            converted_dates = pd.to_datetime(
                cleaned[column],
                errors="coerce",
                utc=column_name.endswith("_datetime")
            )
            if column_name.endswith("_datetime"):
                cleaned[column] = (
                    converted_dates
                    .dt.strftime("%Y-%m-%dT%H:%M:%SZ")
                )
            else:
                cleaned[column] = (
                    converted_dates
                    .dt.strftime("%Y-%m-%d")
                )

        #clean time columns
        elif (
            column_name in TIME_COLUMNS
            or column_name.endswith("_time")
        ):
            cleaned[column] = (
                normalize_text(cleaned[column])
                .str.slice(0, 8)
            )

        #standardize email addresses
        elif (
            column_name in LOWERCASE_COLUMNS
            or "email" in column_name
        ):
            cleaned[column] = (
                normalize_text(cleaned[column])
                .str.lower()
            )

        #standardize state and serial-number values
        elif column_name in UPPERCASE_COLUMNS:
            cleaned[column] = (
                normalize_text(cleaned[column])
                .str.upper()
            )
            
        #format zipcodes
        elif column_name == "zip_code":
            zip_values = pd.to_numeric(
                cleaned[column],
                errors="coerce"
            )
            zip_values = zip_values.astype(
                "Int64"
            )
            zip_values = zip_values.astype(
                "string"
            )
            cleaned[column] = zip_values.str.zfill(
                5
            )

        #standardize status values
        elif (
            column_name in STATUS_COLUMNS
            or column_name.endswith("_status")
        ):
            status_values = (
                normalize_text(cleaned[column])
                .str.lower()
            )
            cleaned[column] = (
                status_values
                .map(STATUS_MAP)
                .fillna(status_values.str.title())
            )

        #standardize capitalization
        elif column_name in TITLE_CASE_COLUMNS:
            cleaned[column] = (
                normalize_text(cleaned[column])
                .str.title()
            )

        #standardize Boolean values
        elif (
            column_name.startswith("is_")
            or column_name.startswith("discontinued")
        ):
            boolean_values = (
                normalize_text(cleaned[column])
                .str.lower()
            )
            cleaned[column] = boolean_values.map(
                {
                    "true": True,
                    "yes": True,
                    "1": True,
                    "false": False,
                    "no": False,
                    "0": False
                }
            )
            
        #convert numeric columns
        elif (
            any(
                keyword in column_name
                for keyword in (
                    "amount",
                    "price",
                    "budget",
                    "rate",
                    "quantity",
                    "hours",
                    "level",
                    "number",
                    "cost",
                    "total"
                )
            )
            and not column_name.endswith("_id")
        ):

            cleaned[column] = pd.to_numeric(
                cleaned[column],
                errors="coerce"
            )
        elif cleaned[column].dtype == "object":

            cleaned[column] = normalize_text(
                cleaned[column]
            )

    #preserve identifier columns as text
    for column in cleaned.columns:

        if (
            column == "id"
            or column.endswith("_id")
        ):

            cleaned[column] = (
                normalize_text(cleaned[column])
                .str.replace(
                    r"\.0$",
                    "",
                    regex=True
                )
            )

    #remove duplicate records
    cleaned = cleaned.drop_duplicates()
    cleaned = cleaned.reset_index(
        drop=True
    )

    #remove rows missing the primary identifier
    if len(cleaned.columns) > 0:

        primary_column = cleaned.columns[0]

        cleaned = cleaned.dropna(
            subset=[primary_column]
        )

    return cleaned

In [6]:
#run the complete ETL process
def main():
    if not INPUT_FILE.exists():

        raise FileNotFoundError(
            "Input workbook not found: "
            f"{INPUT_FILE.resolve()}"
        )

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    workbook = pd.ExcelFile(
        INPUT_FILE
    )

    summary_records = []

    #process all worksheets
    for sheet_name in workbook.sheet_names:

        print(
            f"\nProcessing worksheet: {sheet_name}"
        )

        raw_table = extract_data_table(
            INPUT_FILE,
            sheet_name
        )

        raw_row_count = len(
            raw_table
        )

        cleaned_table = clean_table(
            raw_table
        )

        clean_row_count = len(
            cleaned_table
        )

        #create output filename
        output_filename = (
            f"{snake_case(sheet_name)}.csv"
        )

        #create the full output path
        output_path = (
            OUTPUT_DIR
            / output_filename
        )

        #export the cleaned table

        cleaned_table.to_csv(
            output_path,
            index=False,
            na_rep=""
        )
        rows_removed = (
            raw_row_count
            - clean_row_count
        )
        summary_records.append(
            {
                "table": snake_case(sheet_name),
                "raw_rows": raw_row_count,
                "clean_rows": clean_row_count,
                "rows_removed": rows_removed,
                "output_file": str(output_path)
            }
        )

        print(
            f"{sheet_name}: "
            f"{raw_row_count} raw rows -> "
            f"{clean_row_count} clean rows"
        )

        print(
            f"Exported: {output_path}"
        )

    #create ETL summary report
    summary_dataframe = pd.DataFrame(
        summary_records
    )

    summary_path = (
        OUTPUT_DIR
        / "etl_summary.csv"
    )

    summary_dataframe.to_csv(
        summary_path,
        index=False
    )

    #display the final completion messages

    print(
        "\nETL process completed successfully."
    )

    print(
        "Cleaned CSV files are located in:"
    )

    print(
        OUTPUT_DIR.resolve()
    )

    print(
        "\nETL summary file:"
    )

    print(
        summary_path.resolve()
    )

    print(
        "\nETL Summary:"
    )

    print(
        summary_dataframe.to_string(
            index=False
        )
    )

In [7]:
#run program
if __name__ == "__main__":
    main()


Processing worksheet: Sale Item
Sale Item: 20 raw rows -> 20 clean rows
Exported: checkpoint4_cleaned_files/sale_item.csv

Processing worksheet: Store Expense
Store Expense: 16 raw rows -> 16 clean rows
Exported: checkpoint4_cleaned_files/store_expense.csv

Processing worksheet: Sale
Sale: 20 raw rows -> 20 clean rows
Exported: checkpoint4_cleaned_files/sale.csv

Processing worksheet: Customer
Customer: 18 raw rows -> 18 clean rows
Exported: checkpoint4_cleaned_files/customer.csv

Processing worksheet: Delivery
Delivery: 18 raw rows -> 18 clean rows
Exported: checkpoint4_cleaned_files/delivery.csv

Processing worksheet: Purchase Order Item
Purchase Order Item: 18 raw rows -> 18 clean rows
Exported: checkpoint4_cleaned_files/purchase_order_item.csv

Processing worksheet: Purchase Order
Purchase Order: 18 raw rows -> 18 clean rows
Exported: checkpoint4_cleaned_files/purchase_order.csv

Processing worksheet: Vendor Product
Vendor Product: 20 raw rows -> 20 clean rows
Exported: checkpoint

In [8]:
print("\nFor checkpoint 4 ETL scripts, I used ChatGPT to help identify and correct syntax errors in my code. ChatGPT also provided guidance on understanding and resolving programming logic issues and errors encountered during development.")


For checkpoint 4 ETL scripts, I used ChatGPT to help identify and correct syntax errors in my code. ChatGPT also provided guidance on understanding and resolving programming logic issues and errors encountered during development.
